In [21]:
# Before anything else, you need to create a workflow in the Captain Data UI and copy its UID
# Then, you need to get the API key and the project ID in the Captain Data UI and add them to the .env file
# Then, install the dependencies and you're good to go

import os
import time
import requests
from dotenv import load_dotenv

# This is needed to load environment variables from .env file to the virtual environment
load_dotenv()

API_KEY = os.getenv("CAPTAIN_DATA_API_KEY")
PROJECT_ID = os.getenv("CAPTAIN_DATA_PROJECT_ID")
BASE_URL = "https://api.captaindata.co/v3"

# Verify credentials are loaded
if not API_KEY:
    print("You don't have an API key configured. Set it in the .env file")
else:
    print("API key loaded successfully")

API key loaded successfully


In [22]:
def get_headers():
    """Build request headers with authentication."""
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"x-api-key {API_KEY}",
    }
    if PROJECT_ID:
        headers["x-project-id"] = PROJECT_ID
    return headers


def get_job_status(job_uid: str):
    """
    Get the current status of a job/run.
    
    Args:
        job_uid: The UID of the job to check
    
    Returns:
        Response JSON with job status and details
    """
    url = f"{BASE_URL}/jobs/{job_uid}"
    response = requests.get(url, headers=get_headers())
    response.raise_for_status()
    return response.json()


def get_job_results(job_uid: str):
    """
    Get the results of a completed job.
    
    Args:
        job_uid: The UID of the job to get results for
    
    Returns:
        Response JSON with results data
    """
    url = f"{BASE_URL}/jobs/{job_uid}/results"
    response = requests.get(url, headers=get_headers())
    response.raise_for_status()
    return response.json()


def wait_for_completion(job_uid: str, poll_interval: int = 5, max_wait: int = 300):
    """
    Poll job status until completion or timeout.
    
    Args:
        job_uid: The UID of the job to monitor
        poll_interval: Seconds between status checks
        max_wait: Maximum seconds to wait
    
    Returns:
        Final job status response
    """
    elapsed = 0
    while elapsed < max_wait:
        status = get_job_status(job_uid)
        print(f"Status: {status['status']} (elapsed: {elapsed}s)")
        
        if status["status"] in ["finished", "completed", "shutdown", "warning", "stopped"]:
            return status
        
        time.sleep(poll_interval)
        elapsed += poll_interval
    
    raise TimeoutError(f"Job did not complete within {max_wait} seconds")

In [ ]:
# Put the real workflow UID here. You can get it from the link of the workflow in the platform
WORKFLOW_UID = "2d9439a2-cb9b-4530-b061-513107926ace"

steps = [
    {
        "accounts": [],
        "parameters": {"max_results": 100},
        "output_column": None,
        "step_uid": "b5d88fb9-0c6e-4819-88fd-a8ba9f0eff15"
    },
    {
        "accounts": [],
        "parameters": {},
        "output_column": None,
        "step_uid": "3e3129b5-4316-4a57-b5ac-a98ad914c61f"
    }
]

# Input. For this example, I'm using a template of a workflow that extracts job offers from an Indeed URL
# Note: key must be in lowercase, despite showing as capital letter when configuring the workflow in the UI
inputs = [
    {"indeed_job_search_url": "https://fr.indeed.com/jobs?q=product+owner&l=Puteaux"}
]

# Build the full payload
payload = {
    "steps": steps,
    "unstructure_meta": False,
    "inputs": inputs,
    "job_name": "API Test"
}

# Launch the workflow
try:
    url = f"{BASE_URL}/workflows/{WORKFLOW_UID}/schedule"
    response = requests.post(url, headers=get_headers(), json=payload)
    response.raise_for_status()
    result = response.json()
    job_uid = result.get("job_uid")
    print(f"Workflow launched with Job UID: {job_uid}")
    #print(f"Full response: {result}")
except requests.exceptions.HTTPError as e:
    print(f"Error launching workflow: {e}")
    print(f"Response: {e.response.text if e.response else 'No response'}")

Workflow launched with Job UID: None
Full response: {'template': {'permalink': 'find-companies-recruit-indeed-job-search', 'name': 'Find companies that recruit from Indeed job search', 'uid': 'ae2f32d4-d08c-45f1-8518-b27e750bd4a2'}, 'workflow': {'uid': '2d9439a2-cb9b-4530-b061-513107926ace', 'permalink': 'find-companies-that-recruit-from-indeed-job-search493148e9-9500-4fc3-9b09-d110252131b4', 'name': 'Find companies that recruit from Indeed job search', 'created_at': '2025-12-03T15:30:44.378833', 'project_id': 13749, 'linked_templates': ['26a52c81-024a-42cc-aed7-7281cdb7c393', '4331371d-c651-4d6a-930a-3ff66fd18867'], 'template_uid': 'ae2f32d4-d08c-45f1-8518-b27e750bd4a2'}, 'message': 'Bot successfully scheduled.', 'job_uid': '2d4ceb71-24cc-4e54-b5a8-52027873cb98'}


In [ ]:
# Debug cell - run this to see detailed error info
import json

try:
    url = f"{BASE_URL}/workflows/{WORKFLOW_UID}/schedule"
    response = requests.post(url, headers=get_headers(), json=payload)
    response.raise_for_status()
    result = response.json()
    job_uid = result.get("job_uid")
    print(f"Workflow launched with Job UID: {job_uid}")
    print(f"Full response:\n{json.dumps(result, indent=2)}")
except requests.exceptions.HTTPError as e:
    print(f"Error: {e}")
    print(f"Status code: {e.response.status_code}")
    try:
        error_detail = e.response.json()
        print(f"Response body:\n{json.dumps(error_detail, indent=2)}")
    except:
        print(f"Response text: {e.response.text}")


Workflow launched with Job UID: None
Full response:
{
  "template": {
    "permalink": "find-companies-recruit-indeed-job-search",
    "name": "Find companies that recruit from Indeed job search",
    "uid": "ae2f32d4-d08c-45f1-8518-b27e750bd4a2"
  },
  "workflow": {
    "uid": "2d9439a2-cb9b-4530-b061-513107926ace",
    "permalink": "find-companies-that-recruit-from-indeed-job-search493148e9-9500-4fc3-9b09-d110252131b4",
    "name": "Find companies that recruit from Indeed job search",
    "created_at": "2025-12-03T15:30:44.378833",
    "project_id": 13749,
    "linked_templates": [
      "26a52c81-024a-42cc-aed7-7281cdb7c393",
      "4331371d-c651-4d6a-930a-3ff66fd18867"
    ],
    "template_uid": "ae2f32d4-d08c-45f1-8518-b27e750bd4a2"
  },
  "message": "Bot successfully scheduled.",
  "job_uid": "4b135647-fb63-41a2-9f37-b3a148693ea5"
}


In [ ]:
# wait for job completion

try:
    final_status = wait_for_completion(job_uid, poll_interval=5, max_wait=120)
    print(f"Job completed with status: {final_status['status']}")
    print(f"Workflow: {final_status.get('workflow_name')}")
    print(f"Row count: {final_status.get('row_count')}")
except NameError:
    print("job_uid not defined. Run the previous cells first.")
except TimeoutError as e:
    print(f"TIMEOUT ERROR - Something went wrong when checking status: {e}")
except requests.exceptions.HTTPError as e:
    print(f"ERROR - Something went wrong when checking status: {e}")

Something went wrong when checking status: HTTP error 422 Client Error: unknown for url: https://api.captaindata.co/v3/jobs/None


In [27]:
## Get results from a completed job
try:
    results = get_job_results(job_uid)
    print(f"Results retrieved:")
    print(f"Total items: {results.get('items_count', 0)}")
    print(f"Pages: {results.get('pages', 1)}")
    """   
    # Show the actual results
    if results.get("results"):
        print("\n🔍 Sample data (first 3 items):")
        for i, item in enumerate(results["results"][:3]):
            print(f"\n--- Item {i+1} ---")
            for key, value in item.items():
                # Truncate long values for display
                display_value = str(value)[:100] + "..." if len(str(value)) > 100 else value
                print(f"   {key}: {display_value}")
    else:
        print("No results found")
    """
        
except NameError:
    print("job_uid not defined. Run the previous cells first.")
except requests.exceptions.HTTPError as e:
    print(f"Error fetching results: {e}")

Error fetching results: 422 Client Error: unknown for url: https://api.captaindata.co/v3/jobs/None/results
